In [1]:
# ── Lab 02 - Hub Release and Spaces ──
# Notes : notes/02-hub-release-and-spaces.md
# Result: results/02-release-and-spaces.md
# Goal  : release a model with versions, then create a Space
# Needs : HF_WRITE_TOKEN in the repo-root .env
# Note  : everything made here is deleted in the cleanup cell

# ── imports ──
import os, time
from pathlib import Path
from dotenv import load_dotenv
import huggingface_hub
from huggingface_hub import HfApi, hf_hub_download

# ── load the write token from .env ──
load_dotenv()
WRITE_TOKEN = os.environ["HF_WRITE_TOKEN"]

# ── start clean: no HF_TOKEN ──
os.environ.pop("HF_TOKEN", None)

# ── one API object that uses the write token ──
api = HfApi(token=WRITE_TOKEN)
USER = api.whoami()["name"]

# ── names of the things we will create ──
MODEL_REPO = f"{USER}/release-lab-model"
SPACE_REPO = f"{USER}/release-lab-space"

print("user       :", USER)
print("model repo :", MODEL_REPO)
print("space repo :", SPACE_REPO)

user       : akshayredekar74
model repo : akshayredekar74/release-lab-model
space repo : akshayredekar74/release-lab-space


In [2]:
# ── 1. create a model repo ──
# Private, so nobody else sees it. exist_ok lets you re-run the cell.

api.create_repo(MODEL_REPO, repo_type="model", private=True, exist_ok=True)
print("created: https://huggingface.co/" + MODEL_REPO)

created: https://huggingface.co/akshayredekar74/release-lab-model


In [3]:
# ── 2. upload files + model card ──
# README.md is the model card. The block between --- lines is metadata.

# ── make a local folder with two small files ──
release_dir = Path("release_files")
release_dir.mkdir(exist_ok=True)

(release_dir / "model.bin").write_bytes(b"test weights v1")
(release_dir / "README.md").write_text("""---
license: mit
language: en
tags:
  - test
---

# **Release Lab Model**

A tiny test model used to practise releases.
""")

# ── upload the whole folder in one commit ──
api.upload_folder(
    repo_id=MODEL_REPO,
    folder_path=str(release_dir),
    commit_message="Add model and model card",
)
print("files in repo:", api.list_repo_files(MODEL_REPO))

files in repo: ['.gitattributes', 'README.md', 'model.bin']


In [ ]:
# ── 3. release v1.0 ──
# A tag is a name for one exact commit. It never moves.

api.create_tag(
    MODEL_REPO,
    tag="v1.0",
    tag_message="First release",
)
print("tag v1.0 created")

In [ ]:
# ── 4. change the model, release v1.1 ──

# ── upload a new version of model.bin ──
api.upload_file(
    path_or_fileobj=b"new test weights v2 (better)", # The actual content to upload. 
    # It can be a local file path, raw bytes, or a file-like object.
    path_in_repo="model.bin", # Adds or overwrites model.bin
    repo_id=MODEL_REPO,
    commit_message="Improve weights",
)

# ── tag the new commit ──
api.create_tag(MODEL_REPO, tag="v1.1", tag_message="Better weights")

# ── list all tags ──
refs = api.list_repo_refs(MODEL_REPO)
print("tags:", [t.name for t in refs.tags])

In [3]:
# ── 5. download a specific version ──
# revision= can be a tag, a branch, or a commit id

old_path = hf_hub_download(MODEL_REPO, "model.bin", revision="v1.0", token=WRITE_TOKEN)
new_path = hf_hub_download(MODEL_REPO, "model.bin", revision="v1.1", token=WRITE_TOKEN)

print("v1.0 contains:", Path(old_path).read_bytes())
print("v1.1 contains:", Path(new_path).read_bytes())

v1.0 contains: b'test weights v1'
v1.1 contains: b'new test weights v2 (better)'


In [4]:
# ── 6. work on a branch ──
# A branch is a separate line of work. main stays untouched.

# ── create the branch ──
api.create_branch(MODEL_REPO, branch="dev", exist_ok=True)

# ── upload an experiment to dev only ──
api.upload_file(
    path_or_fileobj=b"experimental notes",
    path_in_repo="experiment.txt",
    repo_id=MODEL_REPO,
    revision="dev",
    commit_message="Add experiment",
)

# ── compare the two branches ──
print("main files:", api.list_repo_files(MODEL_REPO, revision="main"))
print("dev files :", api.list_repo_files(MODEL_REPO, revision="dev"))

No files have been modified since last commit. Skipping to prevent empty commit.


main files: ['.gitattributes', 'README.md', 'model.bin']
dev files : ['.gitattributes', 'README.md', 'experiment.txt', 'model.bin']


In [8]:
# ── 7. commit history ──
# Every upload above made one commit

for c in api.list_repo_commits(MODEL_REPO, revision="main"):
    print(c.commit_id[:8], "|", c.title)

1427e90a | Improve weights
3b71e608 | Add model and model card
0a744997 | initial commit


In [8]:
# ! uv pip install -qU --force-reinstall huggingface_hub gradio

In [21]:
import huggingface_hub
print(huggingface_hub.__file__)
print(huggingface_hub.__version__)

/home/zeus/miniconda3/envs/cloudspace/lib/python3.12/site-packages/huggingface_hub/__init__.py
1.11.0


In [ ]:
import gradio as gr

demo = gr.Interface(
    fn=lambda text: text,
    inputs="text",
    outputs="text",
)

demo.launch(share=True)

In [4]:
api.create_repo(
    SPACE_REPO,
    repo_type="space",
    space_sdk="static",
    private=True,
    exist_ok=True,
)
print("created: https://huggingface.co/spaces/" + SPACE_REPO)

created: https://huggingface.co/spaces/akshayredekar74/release-lab-space


In [5]:
api.upload_file(
    path_or_fileobj="static-space/README.md",
    path_in_repo="README.md",
    repo_id=SPACE_REPO,
    repo_type="space",
    commit_message="Add static Space README",
)

api.upload_file(
    path_or_fileobj="static-space/index.html",
    path_in_repo="index.html",
    repo_id=SPACE_REPO,
    repo_type="space",
    commit_message="Add echo web page",
)

CommitInfo(commit_url='https://huggingface.co/spaces/akshayredekar74/release-lab-space/commit/56fab0373cb53cf3b56c3b7735a142550656e66f', commit_message='Add echo web page', commit_description='', oid='56fab0373cb53cf3b56c3b7735a142550656e66f', pr_url=None, repo_url=RepoUrl('https://huggingface.co/spaces/akshayredekar74/release-lab-space', endpoint='https://huggingface.co', repo_type='space', repo_id='akshayredekar74/release-lab-space'), pr_revision=None, pr_num=None)

In [3]:
# ── 9. upload the app ──
# A Gradio Space needs app.py. requirements.txt is for extra packages.

app_code = '''
import os
import gradio as gr

greeting = os.environ.get("GREETING", "Hello")
secret_ok = "yes" if os.environ.get("MY_SECRET") else "no"

def say_hi(name):
    return f"{greeting}, {name}! (secret is set: {secret_ok})"

gr.Interface(fn=say_hi, inputs="text", outputs="text").launch()
'''

api.upload_file(
    path_or_fileobj=app_code.encode(),
    path_in_repo="app.py",
    repo_id=SPACE_REPO,
    repo_type="space",
    commit_message="Add app",
)
api.upload_file(
    path_or_fileobj=b"",
    path_in_repo="requirements.txt",
    repo_id=SPACE_REPO,
    repo_type="space",
    commit_message="Add empty requirements",
)
print("Space files:", api.list_repo_files(SPACE_REPO, repo_type="space"))

Space files: ['.gitattributes', 'README.md', 'app.py', 'index.html', 'requirements.txt', 'style.css']


In [4]:
# ── 10. variables and secrets ──
# Variable = visible setting. Secret = hidden value (like an API key).

api.add_space_variable(SPACE_REPO, key="GREETING", value="Welcome")
api.add_space_secret(SPACE_REPO, key="MY_SECRET", value="not-a-real-secret")

# ── check what is stored (secret values are never shown) ──
vars_now = api.get_space_variables(SPACE_REPO)
print("variables:", list(vars_now.keys()))

variables: ['GREETING']


In [5]:
# ── 11. wait for the Space ──
# Building takes 1-3 minutes. We check every 15 seconds.

for i in range(16):
    stage = api.get_space_runtime(SPACE_REPO).stage
    print(f"check {i + 1}: {stage}")
    if stage in ("RUNNING", "BUILD_ERROR", "RUNTIME_ERROR", "CONFIG_ERROR"):
        break
    time.sleep(15)

print("open it: https://huggingface.co/spaces/" + SPACE_REPO)

check 1: RUNNING
open it: https://huggingface.co/spaces/akshayredekar74/release-lab-space


In [ ]:
# ── 12. restart the Space ──
# Needed after changing secrets or variables, if it is already running

api.restart_space(SPACE_REPO)
print("restart requested, stage:", api.get_space_runtime(SPACE_REPO).stage)